# Robust Data Agent Evaluation Notebook

This notebook evaluates a Fabric Data Agent against DAX-derived ground truth and then rescores the SDK output with deterministic checks and a local rubric judge.

It runs in five stages:
1. define evaluation questions and compute the expected answers from DAX
2. calibrate the local judge on synthetic pass/fail examples
3. run the Fabric SDK evaluation against the agent
4. rescore each answer with heuristics first and the judge only when needed
5. report split, slice, disagreement, and adjusted metrics

Use this version when you want a more inspectable score than a single binary judge result.


## Package setup

This cell installs the two packages used throughout the notebook: the Fabric Data Agent SDK and the Azure OpenAI client.

Keeping the dependency list small makes the notebook easier to rerun in Fabric while leaving the evaluation logic visible in notebook code.


In [ ]:
%pip install -U fabric-data-agent-sdk openai --q


## Configuration and run settings

This cell defines the workspace, semantic model, data agent, refresh behavior, judge model, and scoring tolerances used by the rest of the notebook.

It also creates run metadata and a unique result table name so each execution is easier to compare later. GPT-5 judge calls intentionally omit an explicit temperature because those endpoints commonly expect default sampling behavior.


In [ ]:
import hashlib
import json
import math
import re
import time
import unicodedata
from datetime import datetime, timezone

import openai
import pandas as pd
import sempy.fabric as fabric
from synapse.ml.fabric.credentials import get_openai_httpx_sync_client
from fabric.dataagent.client import FabricDataAgentManagement

workspace_id = "3261e461-10ce-477e-9b93-824bb2b425b9"
semantic_model_id = "e6b14de2-055f-4dc1-aff5-27f49936b37e"
data_agent_name = "mcaps_eval_dataagent"
data_agent_stage = "production"
workspace_name = None

judge_model = "gpt-5"
fallback_temperature_for_non_gpt5 = 0
numeric_relative_tolerance = 0.01

refresh_model_before_eval = True
refresh_timeout_minutes = 30
refresh_poll_seconds = 15

calibration_repeats = 1
minimum_questions_per_split = 3

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
result_table_name = f"eval_results_{run_id.lower()}"
judge_temperature_mode = (
    "omit_parameter_for_gpt5"
    if judge_model.lower().startswith("gpt-5")
    else f"temperature={fallback_temperature_for_non_gpt5}"
)

run_metadata = {
    "run_id": run_id,
    "workspace_id": workspace_id,
    "semantic_model_id": semantic_model_id,
    "data_agent_name": data_agent_name,
    "data_agent_stage": data_agent_stage,
    "judge_model": judge_model,
    "judge_temperature_mode": judge_temperature_mode,
    "result_table_name": result_table_name,
    "numeric_relative_tolerance": numeric_relative_tolerance,
    "calibration_repeats": calibration_repeats,
}

display(pd.DataFrame([run_metadata]))


## Evaluation set

This section defines the evaluation questions, the DAX used to compute each ground-truth answer, and metadata such as split, slice, and expected answer shape.

The metadata makes later reporting more useful: you can compare dev vs holdout performance, inspect failure patterns by slice, and fingerprint the dataset used for a run.


In [ ]:
eval_cases = [
    {
        "question_id": "dev-approved-orders-count",
        "split": "dev",
        "slice": "scalar",
        "expected_shape": "scalar",
        "question": "How many approved orders are there?",
        "dax": """
DEFINE
  VAR _ApprovedOrders =
    FILTER(
      ALL('orders'),
      'orders'[order_status] = "approved"
    )
EVALUATE
  ROW("Approved Orders", COUNTROWS(_ApprovedOrders))
""",
    },
    {
        "question_id": "dev-customers-in-sao-paulo",
        "split": "dev",
        "slice": "scalar",
        "expected_shape": "scalar",
        "question": "How many customers are in Sao Paulo?",
        "dax": """
EVALUATE
  ROW(
    "Customers in Sao Paulo",
    CALCULATE(
      DISTINCTCOUNT('customers'[customer_id]),
      'customers'[customer_city] = "Sao Paulo"
    )
  )
""",
    },
    {
        "question_id": "dev-top-customer-cities",
        "split": "dev",
        "slice": "ranked_list",
        "expected_shape": "ranked_list",
        "question": "What are the top 3 customer cities by distinct customer count?",
        "dax": """
EVALUATE
  TOPN(
    3,
    SUMMARIZECOLUMNS(
      'customers'[customer_city],
      "Customers", DISTINCTCOUNT('customers'[customer_id])
    ),
    [Customers], DESC,
    'customers'[customer_city], ASC
  )
ORDER BY
  [Customers] DESC,
  'customers'[customer_city] ASC
""",
    },
    {
        "question_id": "holdout-top-customers-sao-paulo",
        "split": "holdout",
        "slice": "ranked_list",
        "expected_shape": "ranked_list",
        "question": "Who are the top 3 customers in Sao Paulo based on number of approved orders?",
        "dax": """
DEFINE
  VAR _SaoPauloCustomers =
    FILTER(
      ALL('customers'),
      'customers'[customer_city] = "Sao Paulo"
    )
  VAR _SummaryTable =
    SUMMARIZECOLUMNS(
      'customers'[customer_id],
      'customers'[customer_city],
      _SaoPauloCustomers,
      "Approved Orders",
        CALCULATE(
          COUNTROWS('orders'),
          'orders'[order_status] = "approved"
        )
    )
EVALUATE
  TOPN(
    3,
    _SummaryTable,
    [Approved Orders], DESC,
    'customers'[customer_id], ASC
  )
ORDER BY
  [Approved Orders] DESC,
  'customers'[customer_id] ASC
""",
    },
    {
        "question_id": "holdout-unfulfilled-city",
        "split": "holdout",
        "slice": "single_entity",
        "expected_shape": "single_entity",
        "question": "Which city had the most customers with unfulfilled orders?",
        "dax": """
DEFINE
  VAR _UnfulfilledByCity =
    SUMMARIZECOLUMNS(
      'customers'[customer_city],
      FILTER(
        ALL('orders'),
        ISBLANK('orders'[order_delivered_customer_date])
      ),
      "Unfulfilled Customers", DISTINCTCOUNT('orders'[customer_id])
    )
EVALUATE
  TOPN(
    1,
    _UnfulfilledByCity,
    [Unfulfilled Customers], DESC,
    'customers'[customer_city], ASC
  )
ORDER BY
  [Unfulfilled Customers] DESC,
  'customers'[customer_city] ASC
""",
    },
    {
        "question_id": "holdout-unfulfilled-orders-count",
        "split": "holdout",
        "slice": "scalar",
        "expected_shape": "scalar",
        "question": "How many orders have not been delivered to the customer?",
        "dax": """
EVALUATE
  ROW(
    "Unfulfilled Orders",
    CALCULATE(
      COUNTROWS('orders'),
      FILTER(
        ALL('orders'),
        ISBLANK('orders'[order_delivered_customer_date])
      )
    )
  )
""",
    },
]

eval_cases_df = pd.DataFrame(eval_cases)
eval_dataset_hash = hashlib.sha256(
    eval_cases_df[["question_id", "split", "slice", "question", "dax"]]
    .to_json(orient="records")
    .encode("utf-8")
).hexdigest()[:16]
run_metadata["eval_dataset_hash"] = eval_dataset_hash
run_metadata["eval_question_count"] = len(eval_cases_df)

display(eval_cases_df[["question_id", "split", "slice", "expected_shape", "question"]])
print(f"Eval dataset hash: {eval_dataset_hash}")

split_counts = eval_cases_df.groupby("split").size().rename("questions").reset_index()
display(split_counts)
if (split_counts["questions"] < minimum_questions_per_split).any():
    print(
        "WARNING: At least one split is still small. Expand each split before treating the score as stable."
    )


## Refresh and ground truth generation

This block optionally refreshes the semantic model, waits for completion, runs each DAX query, and stores the result in both human-readable and structured JSON form.

That gives the notebook a stable truth set for scoring while also recording refresh status and row counts for easier troubleshooting.


In [ ]:
def _to_serializable(value):
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if getattr(type(value), "__module__", "").startswith("numpy"):
        value = value.item()
    if pd.isna(value):
        return None
    return value


def _format_scalar(value):
    value = _to_serializable(value)
    if value is None:
        return "null"
    if isinstance(value, float):
        if math.isclose(value, round(value)):
            return str(int(round(value)))
        return f"{value:.6f}".rstrip("0").rstrip(".")
    return str(value)


def format_dax_result_as_text(df_result):
    if df_result is None or df_result.empty:
        return "No results"

    rows = []
    for _, row in df_result.iterrows():
        parts = [f"{col}: {_format_scalar(row[col])}" for col in df_result.columns]
        rows.append(parts)

    if len(rows) == 1 and len(rows[0]) == 1:
        return rows[0][0].split(": ", 1)[1]

    if len(rows) == 1:
        return ", ".join(rows[0])

    return "\n".join(f"{idx}. " + ", ".join(parts) for idx, parts in enumerate(rows, start=1))


def dataframe_to_structured_truth(df_result):
    if df_result is None or df_result.empty:
        return {"columns": [], "rows": [], "row_count": 0}

    records = []
    for _, row in df_result.iterrows():
        record = {column: _to_serializable(row[column]) for column in df_result.columns}
        records.append(record)

    return {
        "columns": list(df_result.columns),
        "rows": records,
        "row_count": len(records),
    }


def extract_refresh_status(details):
    if isinstance(details, pd.DataFrame) and not details.empty:
        lower_map = {column.lower(): column for column in details.columns}
        for candidate in ["status", "refreshstatus", "state", "extendedstatus"]:
            if candidate in lower_map:
                return str(details.iloc[0][lower_map[candidate]])
    if isinstance(details, dict):
        lower_map = {str(key).lower(): value for key, value in details.items()}
        for candidate in ["status", "refreshstatus", "state", "extendedstatus"]:
            if candidate in lower_map:
                return str(lower_map[candidate])
    return "Unknown"


def wait_for_refresh_completion(
    semantic_model_id,
    refresh_execution_id,
    workspace_id,
    timeout_minutes=30,
    poll_seconds=15,
):
    deadline = time.time() + (timeout_minutes * 60)
    last_details = None

    while time.time() < deadline:
        details = fabric.get_refresh_execution_details(
            semantic_model_id,
            refresh_execution_id,
            workspace_id,
        )
        last_details = details
        status = extract_refresh_status(details).strip().lower()
        print(f"Refresh status: {status}")

        if status in {"completed", "success", "succeeded"}:
            return details
        if status in {"failed", "cancelled", "canceled"}:
            raise RuntimeError(f"Semantic model refresh ended with status: {status}")

        time.sleep(poll_seconds)

    raise TimeoutError("Semantic model refresh did not finish before the timeout.")


if refresh_model_before_eval:
    refresh_execution_id = fabric.refresh_dataset(semantic_model_id, workspace_id)
    refresh_details = wait_for_refresh_completion(
        semantic_model_id=semantic_model_id,
        refresh_execution_id=refresh_execution_id,
        workspace_id=workspace_id,
        timeout_minutes=refresh_timeout_minutes,
        poll_seconds=refresh_poll_seconds,
    )
    run_metadata["refresh_status"] = extract_refresh_status(refresh_details)
else:
    run_metadata["refresh_status"] = "skipped"


dax_outputs = []
for _, row in eval_cases_df.iterrows():
    dax_result = fabric.evaluate_dax(semantic_model_id, row["dax"], workspace_id)
    structured_truth = dataframe_to_structured_truth(dax_result)
    dax_outputs.append(
        {
            "question_id": row["question_id"],
            "expected_answer_text": format_dax_result_as_text(dax_result),
            "ground_truth_json": json.dumps(structured_truth, ensure_ascii=False),
            "ground_truth_row_count": structured_truth["row_count"],
        }
    )

dax_outputs_df = pd.DataFrame(dax_outputs)
eval_cases_df = eval_cases_df.merge(dax_outputs_df, on="question_id", how="left")

display(
    eval_cases_df[
        [
            "question_id",
            "split",
            "slice",
            "question",
            "expected_answer_text",
            "ground_truth_row_count",
        ]
    ]
)


## Service connections

This cell creates the Fabric Data Agent management client and the Azure OpenAI client used for local judging.

Separating connection setup from scoring logic makes it easier to confirm which agent and judge model a run actually used.


In [ ]:
data_agent = FabricDataAgentManagement(data_agent_name, workspace=workspace_id)
judge_client = openai.AzureOpenAI(
    http_client=get_openai_httpx_sync_client(),
    api_version="2025-04-01-preview",
)

print(f"Connected data agent: {data_agent_name}")
print(f"Judge model: {judge_model}")


## Judge prompts and parsing helpers

This section defines the shared judge rules, the short SDK prompt, and the richer local rubric prompt that returns structured JSON.

The helper functions keep prompt construction and response parsing in one place so calibration and rescoring use the same judge behavior.


In [ ]:
judge_rules = f"""
1. Numerical accuracy: values must match within {numeric_relative_tolerance:.0%} relative tolerance unless the query explicitly asks for a rounded number.
2. Entity completeness: every entity in the ground truth must appear in the answer. Missing entities are not allowed.
3. Rank order: for ranked results, the order must match. Correct entities in the wrong rank are incorrect.
4. Semantic equivalence: ignore formatting differences such as markdown, table vs prose, accents, and punctuation.
5. Measure correctness: count, distinct count, average, median, sum, min, and max are not interchangeable.
6. Time scope: a different period, date range, or status filter is incorrect even if the number looks plausible.
7. Extra context: extra non-contradictory context is acceptable.
8. Contradictions: if the answer contains a contradiction, fail it.
9. No-result behavior: if the ground truth is "No results", an answer that clearly states that no matching records were found is acceptable.
10. Refusals: if the answer refuses to answer when the ground truth contains data, fail it.
"""


sdk_critic_prompt = f"""You are an impartial judge evaluating whether a data agent's answer is correct.

Apply these rules in order:
{judge_rules}

Respond with exactly one word: Yes or No.

Query: {{query}}

Ground Truth:
{{expected_answer}}
"""


local_rubric_prompt = f"""You are an impartial judge evaluating whether a data agent's answer is correct.

Apply these rules in order:
{judge_rules}

Return valid JSON only using this schema:
{{{{
  "verdict": "yes|no",
  "reason_code": "exact_match|numeric_mismatch|missing_entity|wrong_rank|wrong_time_scope|wrong_measure|refusal|contradiction|no_results_match|other",
  "numeric_match": true,
  "entity_match": true,
  "rank_match": true,
  "time_scope_match": true,
  "measure_match": true,
  "notes": "short explanation"
}}}}

Query: {{query}}

Ground Truth:
{{expected_answer}}

Actual Answer:
{{actual_answer}}
"""


def build_chat_completion_kwargs(messages):
    kwargs = {
        "model": judge_model,
        "messages": messages,
    }

    if not judge_model.lower().startswith("gpt-5"):
        kwargs["temperature"] = fallback_temperature_for_non_gpt5

    return kwargs


def extract_json_object(text):
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?", "", cleaned).strip()
        cleaned = re.sub(r"```$", "", cleaned).strip()

    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError(f"Judge did not return JSON. Raw output: {text}")

    return json.loads(cleaned[start : end + 1])


def normalize_verdict(value):
    text = str(value).strip().lower()
    if text.startswith("yes"):
        return "yes"
    if text.startswith("no"):
        return "no"
    return "unexpected"


def normalize_boolish(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"true", "yes", "1"}:
        return True
    if text in {"false", "no", "0"}:
        return False
    return None


def run_local_rubric_judge(query, expected_answer, actual_answer):
    prompt = local_rubric_prompt.format(
        query=query,
        expected_answer=expected_answer,
        actual_answer=actual_answer,
    )
    response = judge_client.chat.completions.create(
        **build_chat_completion_kwargs(
            [{"role": "user", "content": prompt}]
        )
    )
    raw_content = response.choices[0].message.content
    payload = extract_json_object(raw_content)
    payload["verdict"] = normalize_verdict(payload.get("verdict"))
    payload["numeric_match"] = normalize_boolish(payload.get("numeric_match"))
    payload["entity_match"] = normalize_boolish(payload.get("entity_match"))
    payload["rank_match"] = normalize_boolish(payload.get("rank_match"))
    payload["time_scope_match"] = normalize_boolish(payload.get("time_scope_match"))
    payload["measure_match"] = normalize_boolish(payload.get("measure_match"))
    payload["raw_response"] = raw_content
    return payload


## Judge calibration cases

This cell creates a small labeled dataset for the local judge, including positive and negative examples across common data-agent failure modes.

The set is intentionally varied so the notebook can measure whether the judge handles numeric answers, entities, ranking, time scope, refusals, and contradictions before using it on real evaluation rows.


In [ ]:
calibration_cases = [
    {
        "case_group": "positive",
        "query": "How many approved orders are there?",
        "expected": "150",
        "actual": "There are 150 approved orders in the dataset.",
        "expected_verdict": "yes",
        "note": "Exact number embedded in prose",
    },
    {
        "case_group": "positive",
        "query": "Which city has the most customers?",
        "expected": "Sao Paulo",
        "actual": "The city with the most customers is São Paulo.",
        "expected_verdict": "yes",
        "note": "Accent equivalence",
    },
    {
        "case_group": "positive",
        "query": "What is the total revenue?",
        "expected": "1,234,567.89",
        "actual": "Total revenue is $1,234,568.",
        "expected_verdict": "yes",
        "note": "Rounded within tolerance",
    },
    {
        "case_group": "positive",
        "query": "Top 3 customers by order count?",
        "expected": "1. customer_id: C001, orders: 15\n2. customer_id: C002, orders: 12\n3. customer_id: C003, orders: 10",
        "actual": "The top 3 customers are C001 with 15 orders, C002 with 12 orders, and C003 with 10 orders.",
        "expected_verdict": "yes",
        "note": "Ranked list preserved in prose",
    },
    {
        "case_group": "positive",
        "query": "What is the first call resolution rate?",
        "expected": "72%",
        "actual": "The first call resolution rate is 0.72.",
        "expected_verdict": "yes",
        "note": "Fraction vs percentage",
    },
    {
        "case_group": "positive",
        "query": "Did any customers match the premium churn rule?",
        "expected": "No results",
        "actual": "No matching records were found for the premium churn rule.",
        "expected_verdict": "yes",
        "note": "No-results equivalence",
    },
    {
        "case_group": "positive",
        "query": "What is the average call handle time?",
        "expected": "4 minutes 32 seconds",
        "actual": "The average call handle time is 4 minutes and 32 seconds. This is above benchmark.",
        "expected_verdict": "yes",
        "note": "Extra non-contradictory context",
    },
    {
        "case_group": "negative",
        "query": "How many approved orders are there?",
        "expected": "150",
        "actual": "There are 250 approved orders.",
        "expected_verdict": "no",
        "note": "Wrong number",
    },
    {
        "case_group": "negative",
        "query": "Which city has the most customers?",
        "expected": "Sao Paulo",
        "actual": "The city with the most customers is Rio de Janeiro.",
        "expected_verdict": "no",
        "note": "Wrong entity",
    },
    {
        "case_group": "negative",
        "query": "Top 3 product categories by complaint volume?",
        "expected": "1. Mobile plans\n2. Broadband\n3. Equipment",
        "actual": "Broadband (1st), Mobile plans (2nd), Equipment (3rd).",
        "expected_verdict": "no",
        "note": "Wrong rank order",
    },
    {
        "case_group": "negative",
        "query": "List all cities with churn rate above 20%.",
        "expected": "North (24%), West (22%), Central (21%)",
        "actual": "North with 24% and West with 22% have churn rates above 20%.",
        "expected_verdict": "no",
        "note": "Missing entity",
    },
    {
        "case_group": "negative",
        "query": "What was the churn rate in Q3?",
        "expected": "Churn rate in Q3 was 14.2%.",
        "actual": "The churn rate was 14.2% in Q4.",
        "expected_verdict": "no",
        "note": "Wrong time scope",
    },
    {
        "case_group": "negative",
        "query": "What is the median call duration?",
        "expected": "6 minutes 10 seconds",
        "actual": "The average call duration is 6 minutes 10 seconds.",
        "expected_verdict": "no",
        "note": "Wrong measure",
    },
    {
        "case_group": "negative",
        "query": "How many calls were flagged for quality review in January?",
        "expected": "317 calls were flagged for quality review in January.",
        "actual": "I was unable to find data on quality review flags for January.",
        "expected_verdict": "no",
        "note": "Refusal when data exists",
    },
    {
        "case_group": "negative",
        "query": "What is the monthly recurring revenue?",
        "expected": "MRR is $2.4M.",
        "actual": "MRR is $2.4M and net revenue retention is 71%.",
        "expected_verdict": "no",
        "note": "Contradictory extra fact",
    },
]

calibration_df = pd.DataFrame(calibration_cases)
display(calibration_df[["case_group", "expected_verdict", "note", "query"]])


## Judge calibration run

This block runs the local judge against every calibration case, captures the structured verdict payload, and computes judge-quality metrics such as accuracy, precision, recall, TPR, and FPR.

Those metrics are later used to decide how much trust to place in judge-scored evaluation rows.


In [ ]:
calibration_results = []

for repeat_index in range(calibration_repeats):
    for _, case in calibration_df.iterrows():
        payload = run_local_rubric_judge(
            query=case["query"],
            expected_answer=case["expected"],
            actual_answer=case["actual"],
        )
        judge_verdict = normalize_verdict(payload.get("verdict"))
        calibration_results.append(
            {
                "repeat": repeat_index + 1,
                "case_group": case["case_group"],
                "query": case["query"],
                "note": case["note"],
                "expected_verdict": case["expected_verdict"],
                "judge_verdict": judge_verdict,
                "correct": judge_verdict == case["expected_verdict"],
                "reason_code": payload.get("reason_code"),
                "numeric_match": payload.get("numeric_match"),
                "entity_match": payload.get("entity_match"),
                "rank_match": payload.get("rank_match"),
                "time_scope_match": payload.get("time_scope_match"),
                "measure_match": payload.get("measure_match"),
                "judge_notes": payload.get("notes"),
            }
        )

judge_cal_df = pd.DataFrame(calibration_results)
display(judge_cal_df)

tp = judge_cal_df[
    (judge_cal_df["expected_verdict"] == "yes")
    & (judge_cal_df["judge_verdict"] == "yes")
].shape[0]
fp = judge_cal_df[
    (judge_cal_df["expected_verdict"] == "no")
    & (judge_cal_df["judge_verdict"] == "yes")
].shape[0]
fn = judge_cal_df[
    (judge_cal_df["expected_verdict"] == "yes")
    & (judge_cal_df["judge_verdict"] == "no")
].shape[0]
tn = judge_cal_df[
    (judge_cal_df["expected_verdict"] == "no")
    & (judge_cal_df["judge_verdict"] == "no")
].shape[0]

total_pos = tp + fn
total_neg = tn + fp

judge_precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
judge_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
judge_f1 = (
    2 * judge_precision * judge_recall / (judge_precision + judge_recall)
    if (judge_precision + judge_recall) > 0
    else 0.0
)
judge_accuracy = judge_cal_df["correct"].mean()
judge_tpr = judge_recall
judge_fpr = fp / total_neg if total_neg > 0 else 0.0

calibration_summary = pd.DataFrame(
    [
        {
            "judge_accuracy": judge_accuracy,
            "judge_precision": judge_precision,
            "judge_recall": judge_recall,
            "judge_f1": judge_f1,
            "judge_tpr": judge_tpr,
            "judge_fpr": judge_fpr,
            "cases": len(judge_cal_df),
        }
    ]
)
display(calibration_summary)

calibration_by_group = (
    judge_cal_df.groupby("case_group")
    .agg(
        cases=("correct", "size"),
        accuracy=("correct", "mean"),
        failures=("correct", lambda series: int((~series).sum())),
    )
    .reset_index()
)
display(calibration_by_group)

calibration_failures = judge_cal_df[~judge_cal_df["correct"]]
if not calibration_failures.empty:
    print("Calibration failures:")
    display(
        calibration_failures[
            ["case_group", "query", "note", "expected_verdict", "judge_verdict", "reason_code", "judge_notes"]
        ]
    )

if judge_accuracy < 0.75:
    print("WARNING: Judge reliability is low. Prefer a fallback judge model or more deterministic scoring.")
elif judge_accuracy < 0.90:
    print("Judge reliability is moderate. Interpret adjusted scores with caution.")
else:
    print("Judge reliability is high enough to use adjusted scoring for non-deterministic cases.")


## SDK evaluation run

This cell sends the evaluation questions to the Fabric Data Agent evaluation API and stores the raw SDK results in a run-specific table.

The SDK output is kept as one scoring signal, not the final word, so later steps can compare it with local rescoring.


In [ ]:
from fabric.dataagent.evaluation import (
    evaluate_data_agent,
    get_evaluation_details,
    get_evaluation_summary,
)

evaluation_request_df = eval_cases_df[["question", "expected_answer_text"]].rename(
    columns={"expected_answer_text": "expected_answer"}
)

evaluation_id = evaluate_data_agent(
    evaluation_request_df,
    data_agent_name,
    workspace_name=workspace_name,
    critic_prompt=sdk_critic_prompt,
    table_name=result_table_name,
    data_agent_stage=data_agent_stage,
)

sdk_summary = get_evaluation_summary(table_name=result_table_name, verbose=False)
eval_details = get_evaluation_details(
    evaluation_id,
    table_name=result_table_name,
    get_all_rows=True,
)

print(f"Evaluation ID: {evaluation_id}")
display(sdk_summary)
display(eval_details)


## Robust rescoring

This section merges the SDK results back to the evaluation metadata, applies deterministic checks first, and calls the local judge only when those checks cannot confidently score a row.

That keeps simple cases cheap and reliable while reserving the judge for answers that need semantic interpretation.


In [ ]:
def find_result_column(df):
    preferred = ["evaluation_result", "eval_result", "result", "is_correct", "pass"]
    for column in preferred:
        if column in df.columns:
            return column

    for column in df.columns:
        lowered = column.lower()
        if "result" in lowered or "eval" in lowered or "pass" in lowered:
            return column

    raise RuntimeError(f"Cannot find evaluation result column. Available columns: {df.columns.tolist()}")


def normalize_binary_result(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"true", "yes", "1", "pass", "passed", "correct"}:
        return True
    if text in {"false", "no", "0", "fail", "failed", "incorrect"}:
        return False
    return None


def strip_accents(text):
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(character for character in normalized if not unicodedata.combining(character))


def normalize_text(text):
    text = strip_accents(str(text)).lower()
    text = re.sub(r"`+", " ", text)
    text = re.sub(r"[^\w\s.%/-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_numeric_candidates(text):
    text = normalize_text(text)
    text = text.replace("$", "").replace("r$", "")

    matches = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?%?|[-+]?\d+(?:\.\d+)?[kmb]", text)
    values = []

    for match in matches:
        token = match.replace(",", "").strip()
        multiplier = 1.0
        is_percent = token.endswith("%")
        suffix = token[-1] if token[-1].isalpha() else ""

        if is_percent:
            token = token[:-1]
        if suffix in {"k", "m", "b"}:
            token = token[:-1]
            multiplier = {"k": 1_000.0, "m": 1_000_000.0, "b": 1_000_000_000.0}[suffix]

        value = float(token) * multiplier
        if is_percent:
            values.append(value)
            values.append(value / 100.0)
        else:
            values.append(value)

    return values


def compare_single_numeric_answer(expected_text, actual_text):
    expected_numbers = extract_numeric_candidates(expected_text)
    actual_numbers = extract_numeric_candidates(actual_text)

    unique_expected = sorted(set(round(number, 10) for number in expected_numbers))
    unique_actual = sorted(set(round(number, 10) for number in actual_numbers))

    if len(unique_expected) != 1 or len(unique_actual) == 0:
        return None

    expected_value = unique_expected[0]
    tolerance = abs(expected_value) * numeric_relative_tolerance
    tolerance = max(tolerance, numeric_relative_tolerance)

    for actual_value in unique_actual:
        if abs(actual_value - expected_value) <= tolerance:
            return True

    if len(unique_actual) == 1:
        return False

    return None


def deterministic_verdict(expected_text, actual_text, expected_shape):
    normalized_expected = normalize_text(expected_text)
    normalized_actual = normalize_text(actual_text)

    if normalized_expected == normalized_actual:
        return {
            "verdict": "yes",
            "reason_code": "heuristic_exact_match",
            "score_source": "heuristic",
        }

    if normalized_expected == "no results":
        no_result_markers = [
            "no results",
            "no matching",
            "no records",
            "unable to find",
            "could not find",
        ]
        if any(marker in normalized_actual for marker in no_result_markers):
            return {
                "verdict": "yes",
                "reason_code": "heuristic_no_results_match",
                "score_source": "heuristic",
            }

    if expected_shape == "scalar":
        numeric_comparison = compare_single_numeric_answer(expected_text, actual_text)
        if numeric_comparison is True:
            return {
                "verdict": "yes",
                "reason_code": "heuristic_numeric_match",
                "score_source": "heuristic",
            }
        if numeric_comparison is False:
            return {
                "verdict": "no",
                "reason_code": "heuristic_numeric_mismatch",
                "score_source": "heuristic",
            }

    return None


result_col = find_result_column(eval_details)
eval_details = eval_details.copy()
eval_details["sdk_pass"] = eval_details[result_col].apply(normalize_binary_result)

if "actual_answer" not in eval_details.columns:
    raise RuntimeError(f"'actual_answer' column not found. Available columns: {eval_details.columns.tolist()}")

robust_eval_df = eval_details.merge(
    eval_cases_df[
        [
            "question_id",
            "split",
            "slice",
            "expected_shape",
            "question",
            "expected_answer_text",
            "ground_truth_json",
        ]
    ],
    on="question",
    how="left",
)

local_scores = []
for _, row in robust_eval_df.iterrows():
    expected_text = str(row["expected_answer_text"])
    actual_text = str(row["actual_answer"])
    heuristic = deterministic_verdict(
        expected_text=expected_text,
        actual_text=actual_text,
        expected_shape=row["expected_shape"],
    )

    if heuristic is not None:
        local_scores.append(
            {
                "robust_verdict": heuristic["verdict"],
                "robust_pass": heuristic["verdict"] == "yes",
                "score_source": heuristic["score_source"],
                "reason_code": heuristic["reason_code"],
                "numeric_match": None,
                "entity_match": None,
                "rank_match": None,
                "time_scope_match": None,
                "measure_match": None,
                "judge_notes": None,
            }
        )
        continue

    payload = run_local_rubric_judge(
        query=row["question"],
        expected_answer=expected_text,
        actual_answer=actual_text,
    )
    local_scores.append(
        {
            "robust_verdict": payload["verdict"],
            "robust_pass": payload["verdict"] == "yes",
            "score_source": "judge",
            "reason_code": payload.get("reason_code"),
            "numeric_match": payload.get("numeric_match"),
            "entity_match": payload.get("entity_match"),
            "rank_match": payload.get("rank_match"),
            "time_scope_match": payload.get("time_scope_match"),
            "measure_match": payload.get("measure_match"),
            "judge_notes": payload.get("notes"),
        }
    )

local_scores_df = pd.DataFrame(local_scores)
robust_eval_df = pd.concat([robust_eval_df.reset_index(drop=True), local_scores_df], axis=1)
robust_eval_df["sdk_vs_robust_match"] = robust_eval_df["sdk_pass"] == robust_eval_df["robust_pass"]

display(
    robust_eval_df[
        [
            "question_id",
            "split",
            "slice",
            "question",
            "expected_answer_text",
            "actual_answer",
            "sdk_pass",
            "robust_pass",
            "score_source",
            "reason_code",
        ]
    ]
)


## Final summaries and adjusted score

This final block turns the per-row rescoring output into run-level summaries: SDK accuracy, robust accuracy, holdout accuracy, split and slice summaries, disagreement rows, and an adjusted score.

The adjusted score only corrects for judge reliability on rows that actually used the judge, which makes the final metric easier to trust and explain.


In [ ]:
total_questions = len(robust_eval_df)
sdk_accuracy = robust_eval_df["sdk_pass"].fillna(False).mean() if total_questions else 0.0
robust_accuracy = robust_eval_df["robust_pass"].mean() if total_questions else 0.0

heuristic_rows = robust_eval_df[robust_eval_df["score_source"] == "heuristic"]
judge_rows = robust_eval_df[robust_eval_df["score_source"] == "judge"]

heuristic_passes = int(heuristic_rows["robust_pass"].sum()) if not heuristic_rows.empty else 0
judge_count = len(judge_rows)
observed_judge_pass_rate = judge_rows["robust_pass"].mean() if judge_count else 0.0

if judge_count and (judge_tpr - judge_fpr) > 0:
    adjusted_judge_pass_rate = (observed_judge_pass_rate - judge_fpr) / (judge_tpr - judge_fpr)
    adjusted_judge_pass_rate = max(0.0, min(1.0, adjusted_judge_pass_rate))
else:
    adjusted_judge_pass_rate = observed_judge_pass_rate

adjusted_total_passes = heuristic_passes + (adjusted_judge_pass_rate * judge_count)
adjusted_accuracy = adjusted_total_passes / total_questions if total_questions else 0.0

judge_margin = ((1 - judge_accuracy) * judge_count / total_questions) if total_questions else 0.0
adjusted_lower_bound = max(0.0, adjusted_accuracy - judge_margin)
adjusted_upper_bound = min(1.0, adjusted_accuracy + judge_margin)

split_summary = (
    robust_eval_df.groupby("split")
    .agg(
        questions=("question", "size"),
        robust_accuracy=("robust_pass", "mean"),
        sdk_accuracy=("sdk_pass", "mean"),
        heuristic_share=("score_source", lambda series: (series == "heuristic").mean()),
        disagreements=("sdk_vs_robust_match", lambda series: int((~series).sum())),
    )
    .reset_index()
)

slice_summary = (
    robust_eval_df.groupby(["split", "slice"])
    .agg(
        questions=("question", "size"),
        robust_accuracy=("robust_pass", "mean"),
        sdk_accuracy=("sdk_pass", "mean"),
        heuristic_share=("score_source", lambda series: (series == "heuristic").mean()),
    )
    .reset_index()
)

disagreement_rows = robust_eval_df[~robust_eval_df["sdk_vs_robust_match"]]

final_summary = pd.DataFrame(
    [
        {
            "run_id": run_id,
            "total_questions": total_questions,
            "sdk_accuracy": sdk_accuracy,
            "robust_accuracy": robust_accuracy,
            "adjusted_accuracy": adjusted_accuracy,
            "adjusted_lower_bound": adjusted_lower_bound,
            "adjusted_upper_bound": adjusted_upper_bound,
            "judge_accuracy": judge_accuracy,
            "judge_scored_questions": judge_count,
            "heuristic_scored_questions": len(heuristic_rows),
            "holdout_accuracy": split_summary.loc[
                split_summary["split"] == "holdout", "robust_accuracy"
            ].iloc[0]
            if (split_summary["split"] == "holdout").any()
            else None,
        }
    ]
)

display(pd.DataFrame([run_metadata]))
display(final_summary)
display(split_summary)
display(slice_summary)

if not disagreement_rows.empty:
    print("Rows where SDK scoring and robust local scoring disagree:")
    display(
        disagreement_rows[
            [
                "question_id",
                "split",
                "slice",
                "question",
                "expected_answer_text",
                "actual_answer",
                "sdk_pass",
                "robust_pass",
                "score_source",
                "reason_code",
                "judge_notes",
            ]
        ]
    )

if (split_summary["questions"] < minimum_questions_per_split).any():
    print("WARNING: One or more splits are still below the minimum sample-size target.")

print(f"SDK accuracy:            {sdk_accuracy:.1%}")
print(f"Robust accuracy:         {robust_accuracy:.1%}")
print(f"Adjusted accuracy:       {adjusted_accuracy:.1%}")
print(f"Adjusted range:          [{adjusted_lower_bound:.1%}, {adjusted_upper_bound:.1%}]")
print(f"Judge-scored questions:  {judge_count}")
print(f"Heuristic-scored rows:   {len(heuristic_rows)}")
